# 第二讲：NumPy 数组运算、向量化操作与随机数生成

**学习目标**
- 掌握逐元素运算和广播机制——NumPy 最核心的「魔法」
- 理解向量化思想，告别 Python 循环
- 熟练使用 ufunc、布尔索引、条件筛选
- 掌握新的 Generator API 进行随机数生成
- 完成一个蒙特卡洛模拟实战，串联全部知识

---

In [2]:
import numpy as np
import time
import sys

## 2.1 数组运算——NumPy 的核心「魔法」

NumPy 数组运算与 Python 列表运算有本质区别。先看一个对比来建立直觉：

In [3]:
# Python 列表的行为
a = [1, 2, 3]
b = [4, 5, 6]

print("Python list:")
print("  a + b =", a + b)       # 拼接！不是逐元素加法
print("  a * 3 =", a * 3)       # 重复！不是逐元素乘法

# NumPy 数组的行为
a_np = np.array([1, 2, 3])
b_np = np.array([4, 5, 6])

print("\nNumPy array:")
print("  a + b =", a_np + b_np)   # 逐元素加法
print("  a * 3 =", a_np * 3)       # 逐元素乘法
print("  a ** 2 =", a_np ** 2)     # 逐元素平方
print("  np.sqrt(a) =", np.sqrt(a_np))  # 逐元素开方

Python list:
  a + b = [1, 2, 3, 4, 5, 6]
  a * 3 = [1, 2, 3, 1, 2, 3, 1, 2, 3]

NumPy array:
  a + b = [5 7 9]
  a * 3 = [3 6 9]
  a ** 2 = [1 4 9]
  np.sqrt(a) = [1.         1.41421356 1.73205081]


### 2.1.1 逐元素运算（Element-wise Operations）

**所有算术运算符都是逐元素的。** 这是 NumPy 最基本也最重要的设计原则。

In [ ]:
a = np.array([1, 2, 3, 4, 5], dtype=np.float64)
b = np.array([10, 20, 30, 40, 50], dtype=np.float64)

print("=== 算术运算 ===")
print("a:", a)
print("b:", b)
print()
print("a + b  =", a + b)
print("a - b  =", a - b)
print("a * b  =", a * b)      # 注意：是逐元素乘，不是矩阵乘法！
print("a / b  =", a / b)
print("a // b =", a // b)     # 整除
print("a % b  =", a % b)      # 取模
print("a ** 2 =", a ** 2)     # 幂运算
print()
print("=== 比较运算（返回布尔数组）===")
print("a > 2:", a > 2)
print("a == b:", a == b)
print("a >= b:", a >= b)
print()
print("=== 与标量的运算（标量自动广播到数组的形状）===")
print("a + 10:", a + 10)
print("a * 2:", a * 2)
print("1 / a:", 1 / a)
print("a > 3:", a > 3)

### 2.1.2 广播机制（Broadcasting）—— NumPy 最强大的特性

广播允许**形状不同**的数组进行运算。理解广播，才算真正掌握了 NumPy。

#### 广播规则（只需记住三条）

1. 从尾部维度开始比较（右对齐）
2. 两个维度**相等**，或其中一个为 **1**，则兼容
3. 不满足条件 → 报错 `ValueError`

#### 图解广播

```
Case 1: (3, 4) 与标量 → 标量广播为 (3, 4)

      4 列                      4 列
  ┌──┬──┬──┬──┐            ┌──┬──┬──┬──┐
3 │  │  │  │  │     +    3 │10│10│10│10│   ← 标量 10 被「拉伸」
  └──┴──┴──┴──┘            └──┴──┴──┴──┘

Case 2: (3, 4) 与 (3, 1) → (3, 1) 广播为 (3, 4)

  (3, 4)      (3, 1)         (3, 4)       (3, 4)
  ┌──┬──┬──┬──┐  ┌──┐    ┌──┬──┬──┬──┐
  │  │  │  │  │  │10│    │10│10│10│10│   ← 列被复制 4 份
  │  │  │  │  │+ │20│ =  │20│20│20│20│
  │  │  │  │  │  │30│    │30│30│30│30│
  └──┴──┴──┴──┘  └──┘    └──┴──┴──┴──┘

Case 3: (3, 4) 与 (4,) → (4,) 广播为 (1, 4) → (3, 4)

  (3, 4)        (4,)            (1, 4)        (3, 4)
  ┌──┬──┬──┬──┐               ┌──┬──┬──┬──┐
  │  │  │  │  │  [1, 2, 3, 4]  →  │ 1│ 2│ 3│ 4│   ← 行被复制 3 份
  │  │  │  │  │+              └──┴──┴──┴──┘
  │  │  │  │  │
  └──┴──┴──┴──┘
```

In [ ]:
# === 广播实战 ===

# 例1：矩阵每一行减去该行的均值（数据中心化）
data = np.array([[1, 2, 3, 4],
                 [5, 6, 7, 8],
                 [9, 10, 11, 12]], dtype=np.float64)

print("原始数据 (3×4):")
print(data)

row_means = data.mean(axis=1, keepdims=True)  # shape (3, 1)
print(f"\n行均值 shape: {row_means.shape}")
print("行均值:")
print(row_means)

centered = data - row_means   # (3,4) - (3,1) → 广播！
print("\n中心化后（每行均值为 0）:")
print(centered)
print(f"\n验证每行均值: {centered.mean(axis=1)}")
print()

# 例2：对每一列加不同的权重
weights = np.array([0.1, 0.2, 0.3, 0.4])  # shape (4,)
weighted = data * weights                    # (3,4) * (4,) → 广播！
print("加权后 (每列乘以不同的系数):")
print(weighted)
print()

# 例3：外积 — 利用广播将一维向量转为二维矩阵
x = np.array([1, 2, 3])           # shape (3,)
y = np.array([10, 20, 30, 40])    # shape (4,)

outer = x[:, np.newaxis] * y[np.newaxis, :]   # (3,1) * (1,4) → (3,4)
print("外积 (3×4):")
print(outer)
print()

# 例4：不兼容的广播 → 报错
try:
    a = np.ones((3, 4))
    b = np.ones((3, 3))   # (3,4) 与 (3,3) → 4≠3，不兼容
    c = a + b
except ValueError as e:
    print(f"广播失败: {e}")

### 2.1.3 聚合运算（Aggregation）—— axis 参数的真相

许多人被 `axis` 参数困扰。记住一个口诀：

> **axis 指定的是「被消灭」的维度。** `axis=0` 意味着沿第 0 轴（行方向）压缩——结果中「行」消失了。`axis=1` 意味着沿第 1 轴（列方向）压缩——结果中「列」消失了。

In [ ]:
# === 图解 axis ===
arr = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])

print("原始数组 (3×3):")
print(arr)
print()

print("axis=0: 沿行方向压缩 → 结果 shape 为 (3,)（行消失，列保留）")
print("  即：每列求一个值")
print("  sum(axis=0):", arr.sum(axis=0), "← 每列的和")
print("  mean(axis=0):", arr.mean(axis=0), "← 每列的均值")
print()

print("axis=1: 沿列方向压缩 → 结果 shape 为 (3,)（列消失，行保留）")
print("  即：每行求一个值")
print("  sum(axis=1):", arr.sum(axis=1), "← 每行的和")
print("  mean(axis=1):", arr.mean(axis=1), "← 每行的均值")
print()

# keepdims: 保留被压缩的维度（方便后续广播）
print("=== keepdims=True 的作用 ===")
without = arr.sum(axis=1)           # shape (3,)
with_kd = arr.sum(axis=1, keepdims=True)  # shape (3, 1)
print(f"  sum(axis=1):            shape={without.shape}")
print(f"  sum(axis=1, keepdims):   shape={with_kd.shape}")
print()
print("有了 keepdims，可以直接用广播做归一化:")
normalized = arr / arr.sum(axis=1, keepdims=True)
print("  每行除以该行和:")
print(normalized)
print(f"  验证每行和: {normalized.sum(axis=1)}")

In [ ]:
# === 常用聚合函数一览 ===
arr = np.array([[1, 3, 5], [2, 4, 6]], dtype=np.float64)

print("数组:")
print(arr, f"shape={arr.shape}")
print()

print(f"sum:    {arr.sum()}")
print(f"prod:   {arr.prod()}")           # 所有元素之积
print(f"mean:   {arr.mean():.2f}")
print(f"std:    {arr.std():.2f}")         # 标准差
print(f"var:    {arr.var():.2f}")         # 方差
print(f"min:    {arr.min()}")
print(f"max:    {arr.max()}")
print(f"argmin: {arr.argmin()}")         # 最小值的位置（扁平化后）
print(f"argmax: {arr.argmax()}")         # 最大值的位置
print(f"cumsum: {arr.cumsum()}")          # 累积和
print(f"cumprod:{arr.cumprod()}")         # 累积积
print()

# === np. 函数形式 vs 数组方法 ===
# 两种写法完全等价，但 np.sum(a, axis=0) 可以用于非 NumPy 序列
a = np.random.randn(3, 4)
print("a.sum() 与 np.sum(a) 完全等价:")
print(f"  a.sum()   = {a.sum():.6f}")
print(f"  np.sum(a) = {np.sum(a):.6f}")

### 2.1.4 线性代数运算

NumPy 提供了丰富的线性代数工具，核心是 `np.dot` 和 `@` 运算符。

**关键区分**：
- `a * b` → 逐元素乘法
- `a @ b` 或 `np.dot(a, b)` → 矩阵乘法

In [ ]:
# === 矩阵乘法 ===
A = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)
B = np.array([[7, 8],
              [9, 10],
              [11, 12]])    # shape (3, 2)

print("矩阵乘法 A @ B (2×3 @ 3×2 → 2×2):")
print(A @ B)
print()

# 等价写法
print("np.dot(A, B) 结果相同:")
print(np.dot(A, B))
print()

print("np.matmul(A, B) 结果相同:")
print(np.matmul(A, B))
print()

# 向量点积
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print(f"向量点积 u·v = {u @ v} = {np.dot(u, v)}")

In [ ]:
# === 更多线性代数工具 ===
from numpy import linalg as LA

M = np.array([[2, 1],
              [1, 2]], dtype=np.float64)

print("矩阵 M:")
print(M)
print()

# 求逆
M_inv = LA.inv(M)
print("M 的逆矩阵:")
print(M_inv)
print(f"验证 M @ M_inv = I:\n{M @ M_inv}")
print()

# 特征值和特征向量
eigenvalues, eigenvectors = LA.eig(M)
print(f"特征值: {eigenvalues}")
print(f"特征向量:\n{eigenvectors}")
print()

# 行列式
print(f"行列式 det(M) = {LA.det(M):.1f}")

# 矩阵范数
print(f"Frobenius 范数: {LA.norm(M, 'fro'):.2f}")

# SVD 分解
U, S, Vt = LA.svd(M)
print(f"SVD: 奇异值 = {S}")

---

## 2.2 向量化操作——告别 Python 循环

### 2.2.1 什么是向量化？

**向量化（Vectorization）** 是指用数组级别的操作代替显式的 Python 循环。

背后的原理：当你在 NumPy 中写 `a + b` 时，循环发生在**编译好的 C 代码层**，而不是 Python 解释器层。Python 解释器每次循环都要做的事——字节码分发、类型检查、方法查找——在 C 层全部免掉了。

```
Python 循环（慢）:                 NumPy 向量化（快）:
                                  
for i in range(len(a)):           c = a + b
    c[i] = a[i] + b[i]            
                                  
↑ Python 层循环 n 次               ↑ C 层循环 1 次
  每次循环都有解释器开销              无 Python 层面开销
```

In [ ]:
# === 向量化到底快多少？——一个直观对比 ===
n = 1_000_000

# Python 显式循环
a = list(range(n))
b = list(range(n))

t0 = time.perf_counter()
c_loop = [a[i] + b[i] for i in range(n)]
t_loop = time.perf_counter() - t0

# NumPy 向量化
a_np = np.arange(n)
b_np = np.arange(n)

t0 = time.perf_counter()
c_vec = a_np + b_np
t_vec = time.perf_counter() - t0

print(f"Python 列表推导式: {t_loop*1000:.1f} ms")
print(f"NumPy 向量化:      {t_vec*1000:.1f} ms")
print(f"加速: {t_loop/t_vec:.0f}×")
print()
print("而且向量化的代码更短、更可读！")

### 2.2.2 通用函数（Universal Functions, ufunc）

ufunc 是 NumPy 中**逐元素操作**的函数。它们都是用 C 实现的，速度极快。

ufunc 分为两类：
- **一元 ufunc**：接受一个数组，返回一个数组（如 `np.sqrt`, `np.exp`, `np.sin`）
- **二元 ufunc**：接受两个数组，返回一个数组（如 `np.add`, `np.multiply`, `np.maximum`）

**每个算术运算符背后都有一个 ufunc**：`+` → `np.add`，`*` → `np.multiply`，等等。

In [ ]:
# === 一元 ufunc ===
x = np.array([0, 1, 2, 3, 10])

print("=== 数学函数 ===")
print("原数组 x:", x)
print("np.sqrt(x):", np.sqrt(x))
print("np.exp(x):", np.exp(x))
print("np.log(x+1):", np.log(x + 1))     # log(0) = -inf，所以用 x+1
print("np.log10(x+1):", np.log10(x + 1))
print()

print("=== 三角函数 ===")
angles = np.array([0, np.pi/2, np.pi])
print("角度:", angles)
print("np.sin:", np.sin(angles))
print("np.cos:", np.cos(angles))
print("np.tan:", np.tan(angles[:2]), "(tan(π/2) → 极大值)")
print()

print("=== 取整/符号 ===")
vals = np.array([-2.7, -1.2, 0.0, 1.2, 2.7])
print("值:", vals)
print("np.floor (向下取整):", np.floor(vals))
print("np.ceil  (向上取整):", np.ceil(vals))
print("np.round (四舍五入):", np.round(vals))
print("np.trunc (向零取整):", np.trunc(vals))
print("np.sign  (符号函数):", np.sign(vals))

In [ ]:
# === 二元 ufunc ===
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("=== 运算符对应的 ufunc ===")
print(f"a + b     = {a + b}        (np.add)")
print(f"np.add(a,b) = {np.add(a,b)}")
print(f"a * b     = {a * b}        (np.multiply)")
print(f"np.multiply(a,b) = {np.multiply(a,b)}")
print()

print("=== 特别的二元 ufunc ===")
print("np.maximum(a, b)  [逐元素取大]:", np.maximum(a, b))
print("np.minimum(a, b)  [逐元素取小]:", np.minimum(a, b))
print("np.power(a, 2)    [逐元素幂]:", np.power(a, 2))
print()

# ufunc 的 out 参数：结果写入已有数组，避免分配新内存
c = np.empty(4)
np.add(a, b, out=c)
print("np.add(a, b, out=c): ", c, "← 结果写入已分配的 c")

### 2.2.3 布尔索引与条件筛选——向量化的 if-else

这是 NumPy 中**最常用也最强大的数据筛选方式**。

In [ ]:
data = np.array([15, 8, 42, 23, 7, 91, 56, 4, 33, 18])
print("原始数据:", data)
print()

# 1. 生成布尔掩码
mask = data > 30
print("data > 30 的掩码:", mask)

# 2. 用掩码筛选
filtered = data[mask]
print("筛选结果 data[mask]:", filtered)
print()

# 3. 组合条件（用 & | ~，不是 and or not！）
print("=== 组合条件 ===")
mask2 = (data > 10) & (data < 50)   # 10 到 50 之间
print("(data > 10) & (data < 50):", data[mask2])

mask3 = (data < 10) | (data > 80)   # 小于10 或 大于80
print("(data < 10) | (data > 80):", data[mask3])

mask4 = ~(data > 30)                # 不大于30（取反）
print("~(data > 30):", data[mask4])
print()

# 4. 直接在索引中编写条件
print("一步筛选 data[data % 2 == 0]:", data[data % 2 == 0], "← 所有偶数")
print()

# 5. 布尔索引赋值（原地修改）
data_copy = data.copy()
data_copy[data_copy > 50] = -1     # 将所有 >50 的元素设为 -1
print("将 >50 的元素设为 -1:", data_copy)

### 2.2.4 np.where / np.select / np.clip —— 向量化的条件逻辑

如果你写 Python 时习惯用 if-elif-else，在 NumPy 中的对应物就是这些函数。

In [ ]:
# === np.where(condition, x, y) ===
# 相当于向量化的: [x_i if condition_i else y_i for each i]

scores = np.array([72, 45, 88, 55, 91, 38])
result = np.where(scores >= 60, "及格", "不及格")
print("成绩:", scores)
print("结果:", result)
print()

# 多层条件 → np.select
conditions = [
    scores >= 90,               # 条件1
    (scores >= 60) & (scores < 90),  # 条件2
    scores < 60                 # 条件3
]
choices = ["优秀", "良好", "不及格"]
grade = np.select(conditions, choices, default="未知")
print("等级:", grade)
print()

# === np.clip —— 限制值的范围 ===
# 将值裁剪到 [lower, upper] 区间内
values = np.array([-5, 0, 5, 10, 15, 100])
clipped = np.clip(values, 0, 10)   # 限制在 [0, 10]
print("原值:", values)
print("裁剪到 [0, 10]:", clipped)
print()

# === np.where 的另一种用法：找到索引 ===
# 省略 x, y 参数时，返回满足条件的元素索引
arr = np.array([10, 20, 30, 40, 50])
indices = np.where(arr > 25)
print("arr > 25 的索引:", indices)
print("arr > 25 的值:", arr[indices])

### 2.2.5 向量化的威力——一个完整对比

下面做一个真实的例子：对 500 万条数据执行标准化操作。

In [ ]:
# 生成 500 万条模拟数据
n = 5_000_000
rng = np.random.default_rng(42)
data = rng.normal(loc=100, scale=15, size=n)  # 均值 100，标准差 15

print(f"数据量: {n:,} 条")
print(f"原始数据: mean={data.mean():.2f}, std={data.std():.2f}")
print()

# ===== 方法1：Python 循环 =====
data_list = data.tolist()
t0 = time.perf_counter()

mean_loop = sum(data_list) / len(data_list)
var_loop = sum((x - mean_loop)**2 for x in data_list) / len(data_list)
std_loop = var_loop ** 0.5
normalized_loop = [(x - mean_loop) / std_loop for x in data_list]

t_loop = time.perf_counter() - t0

# ===== 方法2：NumPy 向量化 =====
t0 = time.perf_counter()

mean_vec = data.mean()
std_vec = data.std()
normalized_vec = (data - mean_vec) / std_vec  # 一行搞定！

t_vec = time.perf_counter() - t0

print("=== 性能对比 ===")
print(f"Python 循环: {t_loop:.2f} s")
print(f"NumPy 向量化: {t_vec:.3f} s")
print(f"加速: {t_loop / t_vec:.0f}×")
print()
print(f"验证标准化结果: mean={normalized_vec.mean():.10f}, std={normalized_vec.std():.2f}")

---

## 2.3 随机数生成——现代 NumPy 的正确用法

### 2.3.1 从 RandomState 到 Generator——为什么要换？

如果你在教程里看到 `np.random.seed()`、`np.random.rand()`、`np.random.randn()`，那是**旧版 API**（`RandomState`）。

**NumPy 1.17 起推荐使用新的 `Generator` API**，原因：

| 特性 | 旧版 RandomState | 新版 Generator |
|------|-----------------|---------------|
| 底层算法 | Mersenne Twister (MT19937) | PCG64 (默认，更快更小) |
| 统计质量 | 部分分布有已知缺陷 | 修复了所有已知缺陷 |
| 并行安全 | 不是线程安全的 | 可通过 SeedSequence 生成多个独立流 |
| 性能 | 较慢 | 更快 |

**规则：所有新代码都应该用 `np.random.default_rng()`。**

In [ ]:
# === 正确用法：创建 Generator 实例 ===
rng = np.random.default_rng(42)  # 设置种子以复现结果
print(f"Generator 类型: {type(rng)}")
print(f"底层 BitGenerator: {rng.bit_generator}")
print()

# === 旧版 API（不推荐，但你需要认识它们）===
np.random.seed(42)             # 全局种子——容易引起副作用
print("旧版 np.random.rand(5):", np.random.rand(5))      # [0, 1) 均匀分布
print("旧版 np.random.randn(5):", np.random.randn(5))    # 标准正态分布
print("旧版 np.random.randint(0,10,5):", np.random.randint(0, 10, 5))
print()
print("⚠️  以上旧版 API 仅用于识别，新代码请用下面的方式！")

### 2.3.2 基础分布——三大常用函数

新版 API 的命名更规范：`integers`（复数）、`uniform`、`choice`。

In [ ]:
rng = np.random.default_rng(42)

# 1. random(size) — [0, 1) 均匀分布（最常用的基础）
print("1. rng.random(10) — [0,1) 均匀分布:")
print("  ", rng.random(10))
print()

# 2. integers(low, high, size) — 随机整数 [low, high)
print("2. rng.integers(1, 7, 10) — 骰子模拟 (1~6):")
print("  ", rng.integers(1, 7, 10))
print(f"  多维: rng.integers(0, 100, (2,3)):\n{rng.integers(0, 100, (2,3))}")
print()

# 3. uniform(low, high, size) — 指定范围的均匀分布
print("3. rng.uniform(-1, 1, 10) — [-1, 1) 均匀分布:")
print("  ", rng.uniform(-1, 1, 10))
print()

# 4. choice(a, size, p) — 从给定数组中随机抽取
colors = np.array(['红', '绿', '蓝', '黄'])
print("4. rng.choice — 随机抽取:")
print("  等概率抽取:", rng.choice(colors, 10))
print("  加权抽取 (红40%绿30%蓝20%黄10%):", 
      rng.choice(colors, 10, p=[0.4, 0.3, 0.2, 0.1]))
print("  不重复抽取:", rng.choice(colors, 3, replace=False))

### 2.3.3 正态分布与统计模拟

正态分布（高斯分布）是数据科学和机器学习中最重要的分布。

In [ ]:
rng = np.random.default_rng(42)

# normal(loc=均值, scale=标准差, size=...) — 通用正态分布
heights = rng.normal(loc=170, scale=7, size=1000)  # 模拟 1000 人身高
print("模拟 1000 人的身高 (cm):")
print(f"  均值: {heights.mean():.1f} cm")
print(f"  标准差: {heights.std():.1f} cm")
print(f"  最低: {heights.min():.1f} cm")
print(f"  最高: {heights.max():.1f} cm")
print()

# standard_normal(size) — 标准正态分布 N(0,1)
z = rng.standard_normal(5)
print("标准正态 N(0,1):", z)
print()

print("=== 更多常用分布 ===")
# 均匀分布 (前面已介绍)
print(f"uniform(-1, 1, 5): {rng.uniform(-1, 1, 5)}")

# 指数分布 (常用于模拟等待时间)
print(f"exponential(scale=2, size=5): {rng.exponential(scale=2, size=5)}")

# 二项分布 (n 次伯努利试验的成功次数)
print(f"binomial(n=10, p=0.5, size=5): {rng.binomial(n=10, p=0.5, size=5)}")

# 泊松分布 (单位时间内随机事件的次数)
print(f"poisson(lam=3, size=5): {rng.poisson(lam=3, size=5)}")

### 2.3.4 种子控制与可复现性

科研和工程中，**可复现性**至关重要。新版 API 提供了强大的工具来管理随机状态。

In [ ]:
# === 基础种子控制 ===
rng1 = np.random.default_rng(42)
rng2 = np.random.default_rng(42)

print("相同种子的两个 Generator 产生相同序列:")
print("  rng1.random(5):", rng1.random(5))
print("  rng2.random(5):", rng2.random(5))
print()

# === SeedSequence: 生成独立但相关的种子 ===
# 适用于并行计算场景（如每个 worker 需要独立的随机流）
ss = np.random.SeedSequence(42)
child_seeds = ss.spawn(3)  # 生成 3 个独立的子种子

rngs = [np.random.default_rng(s) for s in child_seeds]
print("由同一个 SeedSequence 派生的 3 个独立 Generator:")
for i, r in enumerate(rngs):
    print(f"  rng_{i}: {r.random(3)}")
print()

# === 保存与恢复随机状态 ===
rng = np.random.default_rng(42)
state = rng.bit_generator.state       # 保存当前状态
vals1 = rng.random(3)
print(f"第一次 random(3): {vals1}")

rng.bit_generator.state = state       # 恢复状态
vals2 = rng.random(3)
print(f"恢复后 random(3): {vals2}")
print(f"是否相同: {np.allclose(vals1, vals2)}")

### 2.3.5 实战：蒙特卡洛模拟 π 值

用蒙特卡洛方法估算圆周率 π：在一个 2×2 的正方形中撒随机点，统计落在内切圆（半径 1）中的比例。

$$
\frac{\text{圆内点数}}{\text{总点数}} \approx \frac{\pi r^2}{(2r)^2} = \frac{\pi}{4}
\quad\Rightarrow\quad \pi \approx 4 \times \frac{\text{圆内点数}}{\text{总点数}}
$$

这个例子串联了**随机数生成 + 向量化运算 + 布尔索引 + 聚合**，是本讲全部知识的综合应用。

In [ ]:
def monte_carlo_pi(n_points, rng=None):
    """
    用蒙特卡洛方法估算 π。
    
    参数:
        n_points: 撒点数量
        rng: NumPy Generator（可选）
    
    返回:
        估算的 π 值, 误差百分比
    """
    if rng is None:
        rng = np.random.default_rng()
    
    # 1. 在 [-1, 1] × [-1, 1] 正方形内生成随机点
    x = rng.uniform(-1, 1, n_points)   # 向量化！
    y = rng.uniform(-1, 1, n_points)   # 向量化！
    
    # 2. 计算每个点到原点的距离（向量化！）
    distances = np.sqrt(x**2 + y**2)   # 逐元素运算
    
    # 3. 统计落在单位圆内的点（布尔索引 + 聚合！）
    inside = distances <= 1.0          # 布尔掩码
    n_inside = np.sum(inside)          # 聚合：True=1, False=0
    
    # 4. π ≈ 4 × (圆内点数 / 总点数)
    pi_estimate = 4.0 * n_inside / n_points
    
    return pi_estimate, inside


# 运行模拟
rng = np.random.default_rng(42)

for n in [100, 1_000, 10_000, 100_000, 1_000_000]:
    pi_est, _ = monte_carlo_pi(n, rng)
    error = abs(pi_est - np.pi) / np.pi * 100
    print(f"点数: {n:>10,} | π 估算: {pi_est:.10f} | 误差: {error:.6f}%")

In [ ]:
# === 可视化蒙特卡洛过程（用少量点展示） ===
# 如果在 Jupyter 中运行，建议安装 matplotlib 后执行此单元格

rng = np.random.default_rng(42)
pi_est, inside = monte_carlo_pi(5000, rng)

print("5000 个随机点的蒙特卡洛模拟结果:")
print(f"  估算 π = {pi_est:.6f}")
print(f"  真实 π = {np.pi:.6f}")
print(f"  圆内点数: {inside.sum()} / 5000")

try:
    import matplotlib.pyplot as plt
    
    x = rng.uniform(-1, 1, 5000)
    y = rng.uniform(-1, 1, 5000)
    
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_aspect('equal')
    
    # 圆外点（蓝色）、圆内点（红色）
    ax.scatter(x[~inside], y[~inside], s=5, c='steelblue', alpha=0.5, label='圆外')
    ax.scatter(x[inside], y[inside], s=5, c='crimson', alpha=0.5, label='圆内')
    
    # 画单位圆
    theta = np.linspace(0, 2*np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=2, label='单位圆')
    
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_title(f'蒙特卡洛估算 π ≈ {pi_est:.6f} (5000 点)', fontsize=14)
    ax.legend(loc='upper right')
    
    print("\n图表已生成！")
    plt.show()
except ImportError:
    print("\n(需要 matplotlib 来可视化，请运行: pip install matplotlib)")

---

## 本讲小结

| 知识点 | 核心收获 |
|--------|---------|
| **逐元素运算** | 所有算术运算符都是逐元素的；`*` ≠ 矩阵乘法 |
| **广播机制** | 从右对齐比较 shape，尺寸 1 的维度自动扩展 |
| **聚合运算** | `axis` 是「被消灭的维度」；`keepdims=True` 保留维度方便广播 |
| **线性代数** | `@` 和 `np.dot` 做矩阵乘法；`np.linalg` 提供完整工具 |
| **ufunc** | 所有逐元素函数都是 C 实现的；运算符背后就是 ufunc |
| **布尔索引** | `data[mask]` 是最优雅的数据筛选方式；条件用 `&` `|` `~` |
| **条件逻辑** | `np.where` = 向量化 if-else；`np.select` = if-elif-else |
| **向量化思维** | 永远先问自己：「这段循环能不能用数组操作替代？」 |
| **Generator API** | 新代码用 `np.random.default_rng()`，忘掉旧的 `np.random.seed` |
| **可复现性** | 用种子控制随机性；`SeedSequence` 管理并行随机流 |

### 下一步

下一讲将进入 **NumPy 高级操作**：花式索引、结构化数组、内存布局优化、C 顺序与 Fortran 顺序、以及 NumPy 在图像处理中的实际应用。